# Segmentación de clientes para CRM

**Objetivo:** identificar a los clientes interesados en nuevas colecciones, a los recurrentes y a los de mayor valor, y agruparlos en arquetipos para que el equipo de CRM se comunique de forma distinta con cada grupo.

## 1. Carga y vista general

In [1]:
from pathlib import Path
import pandas as pd

RAIZ = Path.cwd().parent
DATA = RAIZ / "data"

ARCHIVOS = ["customers", "orders", "order_items", "products", "stores"]

faltantes = [nombre for nombre in ARCHIVOS if not (DATA / f"{nombre}.csv").exists()]
assert not faltantes, f"Faltan archivos en {DATA}: {faltantes}"

customers   = pd.read_csv(DATA / "customers.csv")
orders      = pd.read_csv(DATA / "orders.csv")
order_items = pd.read_csv(DATA / "order_items.csv")
products    = pd.read_csv(DATA / "products.csv")
stores      = pd.read_csv(DATA / "stores.csv")

tablas = {"customers": customers, "orders": orders, "order_items": order_items,
          "products": products, "stores": stores}

for nombre, df in tablas.items():
    print(f"{nombre}: {df.shape[0]} filas x {df.shape[1]} columnas")

customers: 3122 filas x 10 columnas
orders: 22676 filas x 6 columnas
order_items: 48293 filas x 5 columnas
products: 240 filas x 5 columnas
stores: 5 filas x 2 columnas


In [2]:
for nombre, df in tablas.items():
    print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 3122 entries, 0 to 3121
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   customer_id       3122 non-null   str  
 1   email             3122 non-null   str  
 2   telefono          3112 non-null   str  
 3   fecha_nacimiento  3030 non-null   str  
 4   ciudad            3122 non-null   str  
 5   fecha_registro    3122 non-null   str  
 6   canal_registro    3122 non-null   str  
 7   opt_in_email      3122 non-null   bool 
 8   opt_in_whatsapp   3122 non-null   bool 
 9   notas_crm         165 non-null    str  
dtypes: bool(2), str(8)
memory usage: 201.4 KB
None
<class 'pandas.DataFrame'>
RangeIndex: 22676 entries, 0 to 22675
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   order_id     22676 non-null  str  
 1   customer_id  21876 non-null  str  
 2   fecha        22676 non-null  str  
 3   canal     

**Hallazgos de la estructura**

Se cargaron 5 tablas: 3.122 clientes, 22.676 órdenes, 48.293 ítems de órdenes, 240 productos y 5 tiendas.

1. costumers trae una columna adicional llamada notas_crm
2. Las columnas numéricas (cantidad, precio_unitario, descuento_pct, precio_lista) están bien tipadas como enteros. Que el tipo sea correcto no garantiza que los valores lo sean: se revisan rangos más adelante.
3. descuento_pct es entero lo que sugiere una escala de 0-100, hay que confirmarlo en la siguiente sección
4. las fechas son texto, deben convertirse un revisar si hay formatos mezclados


## 2. Calidad de datos
### 2.1 Duplicados y valores nulos

In [3]:
for nombre, df in tablas.items():
    print(f"\n{nombre}")
    print("  filas duplicadas completas:", df.duplicated().sum())
    print("  nulos por columna:")
    print(df.isna().sum()[df.isna().sum() > 0].to_string())

ids = {"customers": "customer_id", "orders": "order_id",
       "products": "sku", "stores": "tienda_id"}

print("\nIDs repetidos (deberían ser únicos):")
for nombre, col in ids.items():
    print(f"  {nombre}.{col}: {tablas[nombre][col].duplicated().sum()}")


customers
  filas duplicadas completas: 0
  nulos por columna:
telefono              10
fecha_nacimiento      92
notas_crm           2957

orders
  filas duplicadas completas: 0
  nulos por columna:
customer_id      800
tienda_id      12143

order_items
  filas duplicadas completas: 627
  nulos por columna:
Series([], )

products
  filas duplicadas completas: 0
  nulos por columna:
fecha_lanzamiento    24

stores
  filas duplicadas completas: 0
  nulos por columna:
Series([], )

IDs repetidos (deberían ser únicos):
  customers.customer_id: 0
  orders.order_id: 0
  products.sku: 0
  stores.tienda_id: 0


**Hallazgos de duplicados y nulos**

| Tabla | Hallazgo | Hipótesis / impacto |
|---|---|---|
| customers | telefono (10) y fecha_nacimiento (92) nulos | Bajo impacto: no se usan para responder las tres preguntas, sin embargo sirven para revisar la integridad de otras columnas |
| customers | notas_crm vacía en 2.957 de 3.122 clientes (95%) | Solo 165 clientes tienen notas sin embargo algunas de las notas nos dan información que puede aportar de manera significativa al analisis |
| orders | 800 ordenes sin customer_id | Son ventas reales (tienen order_id), pero no se pueden atribuir a un cliente. Cuentan en el total de ventas, no en el análisis por cliente. |
| orders | 12.143 órdenes sin tienda_id | Probablemente son las órdenes online, que no tienen tienda física. |
| order_items | 627 filas duplicadas completas | Posible doble carga. Si no se corrige, infla las ventas y el valor de algunos clientes. |
| products | 24 productos sin fecha_lanzamiento | Verificado: todos son BASICOS, que no tienen temporada. No es un error. |


No hay IDs repetidos en ninguna tabla. Sin embargo, un ID único no garantiza que cada cliente sea una persona distinta: se revisa por email tambien

### 2.2 Plan de verificación

### Hallazgos iniciales y plan de verificación

| Código | Tabla | Análisis | Qué se revisa | Acción si se confirma |
|---|---|---|---|---|
| C1 | customers | Duplicados por email | Mismo email (normalizado) con distinto customer_id | Unificar al cliente más antiguo y reasignar sus órdenes |
| C2 | customers | Formato de email | Emails sin estructura válida | Marcar como no contactable por email |
| C3 | customers | Consistencia de texto | ciudad y canal_registro escritos de varias formas | Normalizar (minúsculas, espacios, sinónimos) |
| C4 | customers | Fechas de cliente | Formatos mezclados, registro futuro, edad < 15 o > 100 | Convertir; los valores imposibles pasan a nulo |
| C5 | customers | Consentimiento vs. contacto | Opt-in activo sin teléfono o con email inválido | Marcar como no activable en ese canal |
| C6 | customers | Contenido de notas_crm | Etiquetas útiles vs. textos con forma de instrucción | Etiquetas como contexto; los lineamientos se validan con el negocio antes de usarlos |
| O1 | orders | Consistencia de texto | canal y estado | Normalizar si hay variantes |
| O2 | orders | Canal vs. tienda | Tienda sin tienda_id u online con tienda_id | Corregir o documentar |
| O3 | orders | Integridad referencial | customer_id que no existe en customers, tienda_id que no existe en stores | Excluir del análisis por cliente; se mantienen en el total |
| O4 | orders | Fechas de orden | Formatos, rango, fechas futuras, órdenes antes del registro | Convertir; documentar los casos imposibles |
| O5 | orders | Distribución de estados | % de completadas, devueltas y canceladas | Solo las completadas cuentan como venta |
| I1 | order_items | Duplicados exactos | 627 filas idénticas en todas las columnas | Pendiente de confirmar con el negocio; se calcula el impacto con y sin ellas |
| I2 | order_items | Rangos numéricos | cantidad ≤ 0, precio_unitario ≤ 0, descuentos irreales | Excluir o corregir según el caso |
| I3 | order_items | Integridad referencial | order_id que no existe en orders, sku que no existe en products | Excluir los ítems huérfanos |
| I4 | order_items | Precio vs. precio de lista | Precios muy alejados de precio_lista | Revisar si es un error de escala |
| P1 | products | Fechas de lanzamiento nulas | 24 productos sin fecha | Verificado: todos son BASICOS, sin temporada. No es un error. |
| P2 | products | Consistencia de colección | Nombre de colección, categoría y fecha de lanzamiento coherentes con la temporada | Normalizar y documentar las incoherencias |
| X1 | entre tablas | Órdenes sin ítems | Órdenes que no tienen ninguna línea en order_items | Documentar; no generan venta |

### 2.3 Verificación y limpieza

In [4]:
customers_clean   = customers.copy()
orders_clean      = orders.copy()
order_items_clean = order_items.copy()
products_clean    = products.copy()

registro = []

def registrar(codigo, tabla, problema, filas, accion):
    """Anota un hallazgo de calidad para la tabla resumen final."""
    registro.append({"codigo": codigo, "tabla": tabla, "problema": problema,
                     "filas_afectadas": filas, "accion": accion})

#### C1. Duplicados por email

In [5]:
email_norm = customers_clean["email"].str.strip().str.lower()

print("Repetidos tal cual:", customers_clean["email"].duplicated().sum())
print("Repetidos al normalizar:", email_norm.duplicated().sum())

# ¿Cuántas personas tienen más de un registro?
conteo = email_norm.value_counts()
print("Emails con más de un customer_id:", (conteo > 1).sum())
print("Máximo de registros para un mismo email:", conteo.max())

Repetidos tal cual: 0
Repetidos al normalizar: 116
Emails con más de un customer_id: 116
Máximo de registros para un mismo email: 2


In [6]:
dup_mask = email_norm.duplicated(keep=False)   # marca AMBOS registros de cada par
pares = customers_clean[dup_mask].assign(email_norm=email_norm[dup_mask])

# ¿Cómo se registró cada par? (ej. "online + tienda")
print(pares.groupby("email_norm")["canal_registro"]
           .apply(lambda s: " + ".join(sorted(s))).value_counts())

# ¿Los dos registros tienen compras?
ids_dup = pares["customer_id"]
print("\nIDs duplicados con órdenes:", ids_dup.isin(orders["customer_id"]).sum(), "de", len(ids_dup))

canal_registro
online + tienda    116
Name: count, dtype: int64

IDs duplicados con órdenes: 175 de 232


In [7]:
ordenes_afectadas = orders["customer_id"].isin(ids_dup).sum()
ambos = (pares.assign(compro=pares["customer_id"].isin(orders["customer_id"]))
              .groupby("email_norm")["compro"].all().sum())
print("Órdenes de clientes duplicados:", ordenes_afectadas)
print("Pares donde ambos registros tienen compras:", ambos)

Órdenes de clientes duplicados: 722
Pares donde ambos registros tienen compras: 65


In [8]:
cc = customers_clean.assign(
    email_norm=email_norm,
    fecha_registro_dt=pd.to_datetime(customers_clean["fecha_registro"], format="mixed", errors="coerce"),
).sort_values(["email_norm", "fecha_registro_dt"])          # de más antiguo a más reciente

# Datos que se calculan sobre el par completo
cc["omnicanal"] = cc.groupby("email_norm")["canal_registro"].transform("nunique") > 1
cc[["opt_in_email", "opt_in_whatsapp"]] = (
    cc.groupby("email_norm")[["opt_in_email", "opt_in_whatsapp"]].transform("all")
)

# Identidad: el registro más antiguo (ID, fecha y canal de registro)
cc["id_maestro"] = cc.groupby("email_norm")["customer_id"].transform("first")
mapa_ids = dict(zip(cc["customer_id"], cc["id_maestro"]))
maestros = cc[cc["customer_id"] == cc["id_maestro"]].set_index("email_norm")

# Contacto: el dato más reciente disponible (last ignora los nulos)
campos_contacto = ["telefono", "fecha_nacimiento", "ciudad", "notas_crm"]
maestros.update(cc.groupby("email_norm")[campos_contacto].last())
maestros["email"] = maestros.index                          # email ya normalizado

customers_clean = maestros.reset_index(drop=True).drop(columns=["id_maestro", "fecha_registro_dt"])
orders_clean["customer_id"] = orders_clean["customer_id"].map(mapa_ids).fillna(orders_clean["customer_id"])

registrar("C1", "customers", "Misma persona registrada en tienda y online con dos customer_id",
          116, "Se conserva el registro más antiguo y se toman los datos de contacto más recientes; "
               "las órdenes del duplicado se reasignan")

print("Clientes después de unificar:", len(customers_clean))

Clientes después de unificar: 3006


In [9]:
ids_eliminados = set(ids_dup) - set(customers_clean["customer_id"])
print("Clientes:", len(customers), "->", len(customers_clean))
print("IDs eliminados:", len(ids_eliminados))
print("Órdenes que aún apuntan a un ID eliminado:", orders_clean["customer_id"].isin(ids_eliminados).sum())
print("Total de órdenes sin cambio:", len(orders) == len(orders_clean))

Clientes: 3122 -> 3006
IDs eliminados: 116
Órdenes que aún apuntan a un ID eliminado: 0
Total de órdenes sin cambio: True


**Problema:** 116 personas están registradas dos veces, con el mismo email escrito con distinta capitalización o espacios y dos customer_id distintos. En los 116 casos un registro es de tienda y el otro online: los canales no comparten un perfil único de cliente.

**Tratamiento:** se normalizó el email (minúsculas y sin espacios), se conservó el registro más antiguo de cada persona y se le reasignaron las X órdenes de su duplicado. Las órdenes no se eliminan; solo cambian de dueño. El consentimiento de contacto queda activo solo si estaba activo en ambos registros. Resultado: 3.006 clientes, en línea con los aproximados 3.000 que describe el caso.

**Impacto si no se corrige:** en Y casos ambos registros tienen compras, así que el historial de esas personas queda partido en dos perfiles. Se verían menos recurrentes y de menor valor de lo que realmente son, lo que afecta directamente las preguntas 1, 2 y 3.

#### C2. Formato de email

El email es el canal principal para las campañas de CRM. Un email mal formado no se puede contactar, aunque el cliente haya dado su consentimiento. Se revisa que cada email tenga la estructura mínima usuario@dominio.extensión.

In [10]:
patron = r"^[^@\s]+@[^@\s]+\.[^@\s]+$"
customers_clean["email_valido"] = customers_clean["email"].str.match(patron)

invalidos = customers_clean.loc[~customers_clean["email_valido"], "email"]
print("Emails inválidos:", len(invalidos), f"({len(invalidos) / len(customers_clean):.1%})")

# ¿Qué tipo de error tienen? (sin mostrar los emails)
print("Sin @:", (~invalidos.str.contains("@")).sum())
print("Más de una @:", (invalidos.str.count("@") > 1).sum())
print("Sin punto en el dominio:", (invalidos.str.contains("@") & ~invalidos.str.split("@").str[-1].str.contains(r"\.")).sum())
print("Con espacios internos:", invalidos.str.contains(r"\s").sum())

print("\nDe los inválidos, cuántos tienen opt-in de email activo:",
      customers_clean.loc[~customers_clean["email_valido"], "opt_in_email"].sum())

Emails inválidos: 8 (0.3%)
Sin @: 2
Más de una @: 2
Sin punto en el dominio: 4
Con espacios internos: 0

De los inválidos, cuántos tienen opt-in de email activo: 8


In [11]:
registrar("C2", "customers", "Email con formato inválido (sin @, doble @ o sin dominio)", 8,
          "No se corrige para no adivinar el contacto; se marca email_valido = False")

**Problema:** 8 clientes (0,3%) tienen un email inválido: 2 sin @, 2 con más de una @ y 4 sin punto en el dominio. Los 8 tienen activo el consentimiento para recibir emails.

**Tratamiento:** no se corrigen automáticamente. Adivinar la dirección correcta podría enviar comunicaciones a otra persona, lo que es un riesgo de privacidad. Se conservan en el análisis (sus compras son válidas) y se marcan como no contactables por email.

**Impacto si no se corrige:** campañas con rebote y una base contactable inflada. Por su bajo volumen no afecta las conclusiones del análisis.

**Recomendación:** validar el formato del email en el momento del registro y solicitar la actualización a estos 8 clientes en su próxima compra.

#### C3. Consistencia de texto

ciudad y canal_registro se usan para agrupar clientes. Si un mismo valor está escrito de varias formas (mayúsculas, tildes, abreviaturas), un grupo se parte en varios y los conteos y ventas por ciudad o canal salen incorrectos.

In [12]:
for col in ["ciudad", "canal_registro"]:
    print(f"\n{col}:")
    print(customers_clean[col].value_counts(dropna=False).to_string())


ciudad:
ciudad
León                411
Monterrey           407
Querétaro           384
Guadalajara         371
Tijuana             365
Puebla              360
Mérida              358
Ciudad de México    350

canal_registro:
canal_registro
online    1634
tienda    1372


**Resultado:** sin hallazgos. Las 8 ciudades y los 2 canales de registro están escritos de forma consistente. Los clientes se reparten de forma similar entre ciudades (entre 350 y 411 cada una) y entre canales (1.634 online y 1.372 en tienda).

#### C4. Fechas de cliente

La fecha de registro define la antigüedad del cliente y la de nacimiento su edad. Si hay formatos mezclados, pandas puede leer mal las fechas (por ejemplo, confundir día y mes) o no leerlas. Primero se identifican los formatos presentes, reemplazando cada dígito por 9 para ver la forma de la fecha sin exponer el dato.

In [13]:
for col in ["fecha_registro", "fecha_nacimiento"]:
    formatos = customers_clean[col].str.replace(r"\d", "9", regex=True)
    print(f"\nFormatos de {col}:")
    print(formatos.value_counts(dropna=False).to_string())


Formatos de fecha_registro:
fecha_registro
9999-99-99    3006

Formatos de fecha_nacimiento:
fecha_nacimiento
9999-99-99    2923
NaN             83


Todas las fechas tienen la forma AAAA-XX-XX, sin formatos mezclados. Falta confirmar si es año-mes-día o año-día-mes: si la parte del medio supera 12 en algún caso, no puede ser un mes. Luego se convierten de texto a fecha con el formato explícito y se revisa que los valores tengan sentido.

In [14]:
for col in ["fecha_registro", "fecha_nacimiento"]:
    partes = customers_clean[col].str.split("-", expand=True).astype(float)
    print(f"{col}: parte del medio máx = {partes[1].max():.0f}, parte final máx = {partes[2].max():.0f}")

for col in ["fecha_registro", "fecha_nacimiento"]:
    convertida = pd.to_datetime(customers_clean[col], format="%Y-%m-%d", errors="coerce")
    no_convertidas = convertida.isna().sum() - customers_clean[col].isna().sum()
    print(f"\n{col}: {no_convertidas} fechas inválidas | rango {convertida.min().date()} a {convertida.max().date()}")
    customers_clean[col] = convertida

hoy = pd.Timestamp("2026-10-14")
edad = (hoy - customers_clean["fecha_nacimiento"]).dt.days / 365.25
print("\nRegistros en el futuro:", (customers_clean["fecha_registro"] > hoy).sum())
print("Edad < 15:", (edad < 15).sum(), "| Edad > 100:", (edad > 100).sum())
print("Registrado antes de nacer:", (customers_clean["fecha_registro"] < customers_clean["fecha_nacimiento"]).sum())

fecha_registro: parte del medio máx = 12, parte final máx = 31
fecha_nacimiento: parte del medio máx = 12, parte final máx = 28

fecha_registro: 0 fechas inválidas | rango 2022-01-01 a 2026-06-30

fecha_nacimiento: 0 fechas inválidas | rango 1900-01-01 a 2031-05-04

Registros en el futuro: 0
Edad < 15: 8 | Edad > 100: 34
Registrado antes de nacer: 8


La parte del medio nunca supera 12: el formato es año-mes-día, sin ambigüedad. Todas las fechas se convierten correctamente y no hay registros futuros. Sin embargo, la fecha de nacimiento va de 1900-01-01 a 2031-05-04: hay 34 clientes con más de 100 años y 8 con fecha de nacimiento futura.

In [15]:
print("Exactamente 1900-01-01:", (customers_clean["fecha_nacimiento"] == "1900-01-01").sum())

imposible = (edad > 100) | (edad < 15)
customers_clean.loc[imposible, "fecha_nacimiento"] = pd.NaT

registrar("C4", "customers", "Fecha de nacimiento imposible (valor por defecto 1900-01-01 o fecha futura)",
          int(imposible.sum()), "Se reemplaza por nulo; no se usa la edad en esos clientes")

print("Fechas de nacimiento anuladas:", imposible.sum())

Exactamente 1900-01-01: 34
Fechas de nacimiento anuladas: 42


**Resultado:** todas las fechas usan el formato año-mes-día y se convirtieron de texto a tipo fecha. No hay fechas de registro futuras; los registros van de 2022-01-01 a 2026-06-30.

**Problema:** 42 fechas de nacimiento son imposibles: 34 con edad mayor de 100 años (principalmente 1900-01-01, un valor por defecto del sistema) y 8 en el futuro (hasta 2031).

**Tratamiento:** se reemplazan por nulo. Inventar una edad metería datos falsos en el perfil de los arquetipos; es mejor un dato faltante que uno incorrecto.

#### C5. Consentimiento vs. datos de contacto

Para que una campaña llegue, el cliente necesita el consentimiento activo y un dato de contacto válido en ese canal. Se revisan los clientes con consentimiento pero sin un contacto utilizable, y el formato de los teléfonos (en México un celular tiene 10 dígitos, o 12 con el prefijo 52).

In [16]:
c = customers_clean
tel_digitos = c["telefono"].str.replace(r"\D", "", regex=True)

print("Opt-in WhatsApp sin teléfono:", (c["opt_in_whatsapp"] & c["telefono"].isna()).sum())
print("Opt-in email con email inválido:", (c["opt_in_email"] & ~c["email_valido"]).sum())

print("\nLongitud de los teléfonos (solo dígitos):")
print(tel_digitos.str.len().value_counts(dropna=False).sort_index().to_string())

Opt-in WhatsApp sin teléfono: 8
Opt-in email con email inválido: 8

Longitud de los teléfonos (solo dígitos):
telefono
2.0        4
4.0        4
10.0      48
12.0    2940
NaN       10


La mayoría de teléfonos tiene 12 dígitos (prefijo 52 + 10 dígitos). Hay 48 con 10 dígitos, válidos pero sin prefijo, y 8 con 2 o 4 dígitos, que no son un teléfono válido. Además, 8 clientes tienen opt-in de WhatsApp sin teléfono y 8 tienen opt-in de email con un email inválido (los identificados en C2).

In [17]:
# Estandarizar al formato 52 + 10 dígitos
tel = customers_clean["telefono"].str.replace(r"\D", "", regex=True)
tel = tel.where(tel.str.len() != 10, "52" + tel)            # a los de 10 dígitos se les agrega el prefijo
valido = tel.str.len().eq(12) & tel.str.startswith("52")

customers_clean["telefono"] = tel.where(valido)             # los inválidos quedan como nulo
customers_clean["telefono_valido"] = valido.fillna(False)

# Contactable = consentimiento + dato válido en ese canal
customers_clean["contactable_email"]    = customers_clean["opt_in_email"] & customers_clean["email_valido"]
customers_clean["contactable_whatsapp"] = customers_clean["opt_in_whatsapp"] & customers_clean["telefono_valido"]

registrar("C5", "customers", "Teléfono sin prefijo de país (10 dígitos)", 48,
          "Se estandariza agregando el prefijo 52")
registrar("C5", "customers", "Teléfono incompleto (2 o 4 dígitos)", 8,
          "Se reemplaza por nulo y se marca como no contactable por WhatsApp")

print("Opt-in WhatsApp no contactables:", (customers_clean["opt_in_whatsapp"] & ~customers_clean["telefono_valido"]).sum())
print("Opt-in email no contactables:", (customers_clean["opt_in_email"] & ~customers_clean["email_valido"]).sum())

Opt-in WhatsApp no contactables: 16
Opt-in email no contactables: 8


**Problema:** 48 teléfonos no tenían prefijo de país y 8 estaban incompletos. Además, algunos clientes tienen consentimiento activo sin un dato de contacto válido en ese canal.

**Tratamiento:** los teléfonos de 10 dígitos se estandarizaron con el prefijo 52; los incompletos se reemplazaron por nulo. No se modificó el consentimiento del cliente: se crearon las columnas contactable_email y contactable_whatsapp, que combinan consentimiento y dato válido.

**Impacto si no se corrige:** se sobrestimaría la base contactable y las campañas intentarían envíos que fallan. Con formatos de teléfono distintos, además, la herramienta de envío podría rechazar números válidos.

#### C6. Contenido de notas_crm

notas_crm es una columna de texto libre que no aparece en la documentación de los datos y solo está llena en 165 clientes. Antes de decidir si se usa, se revisa qué tipo de información contiene.

In [18]:
print(customers_clean["notas_crm"].value_counts(dropna=True).to_string())

notas_crm
Actualizó datos en tienda                                                                                                                                                                             27
Cliente frecuente de temporada                                                                                                                                                                        23
Revisar teléfono                                                                                                                                                                                      23
Cambio de dirección reciente                                                                                                                                                                          22
Prefiere contacto por WhatsApp                                                                                                                                                            

Las notas son 10 textos fijos que se repiten (plantillas), y se dividen en tres grupos:

- **Señales útiles:** "Registro en evento de lanzamiento" y "Cliente frecuente de temporada" sirven para validar los criterios de las preguntas 1 y 2. "Prefiere contacto por WhatsApp", "Pidió no recibir llamadas" y "Revisar teléfono" son preferencias y alertas de contacto.
- **Operativas:** "Solicitó factura", "Actualizó datos en tienda" y "Cambio de dirección reciente" no aportan al análisis.
- **Textos con forma de instrucción:** un "Lineamiento CRM 2026" que define nombres y criterios de segmentación, y una "Nota de exportación" que agrupa Tijuana y Mérida en una misma región, aunque están en extremos opuestos del país. No son fuentes oficiales de definiciones de negocio.

In [19]:
notas = customers_clean["notas_crm"]

customers_clean["nota_evento_lanzamiento"]  = notas.eq("Registro en evento de lanzamiento").fillna(False)
customers_clean["nota_frecuente_temporada"] = notas.eq("Cliente frecuente de temporada").fillna(False)
customers_clean["nota_prefiere_whatsapp"]   = notas.eq("Prefiere contacto por WhatsApp").fillna(False)
customers_clean["nota_no_llamadas"]         = notas.eq("Pidió no recibir llamadas").fillna(False)
customers_clean["nota_revisar_telefono"]    = notas.eq("Revisar teléfono").fillna(False)

instruccion = notas.str.startswith(("Lineamiento", "Nota de exportación"), na=False)
registrar("C6", "customers", "Texto con forma de instrucción de negocio dentro de notas_crm",
          int(instruccion.sum()), "No se aplica; se documenta como hallazgo y la columna no se envía a modelos externos")

customers_clean = customers_clean.drop(columns=["notas_crm"])
print("Clientes con texto tipo instrucción:", instruccion.sum())

Clientes con texto tipo instrucción: 3


**Problema:** notas_crm mezcla información útil con textos que parecen instrucciones de negocio (nombres de segmentos, ventanas de recurrencia y agrupaciones de ciudades). Si se tomaran como reglas, cambiarían los criterios del análisis sin ningún respaldo.

**Tratamiento:** las notas útiles se convirtieron en columnas booleanas; los textos tipo instrucción no se aplican. No tienen una fuente oficial que los respalde, así que las definiciones y criterios del análisis son propios. La columna de texto libre se eliminó.

**Impacto si no se corrige:** se habrían aplicado criterios sin fuente oficial y se habrían mezclado ciudades de regiones distintas. Además, si este texto se procesara con un modelo de IA sin filtrar, las instrucciones podrían alterar los resultados sin que nadie lo notara.


#### O1. Consistencia de texto

canal y estado definen qué órdenes cuentan como venta y por qué canal se hicieron. Si un valor está escrito de varias formas, las órdenes se reparten en categorías falsas y los filtros (por ejemplo, "solo completadas") dejan órdenes por fuera.

In [20]:
for col in ["canal", "estado"]:
    print(f"\n{col}:")
    print(orders_clean[col].value_counts(dropna=False).to_string())


canal:
canal
online    12143
tienda    10533

estado:
estado
completada    21322
devuelta        898
cancelada       456


**Resultado:** sin hallazgos. canal tiene 2 valores (online: 12.143; tienda: 10.533) y estado tiene 3 (completada: 21.322; devuelta: 898; cancelada: 456), todos escritos de forma consistente.

#### O2. Canal vs. tienda

Una orden online no debería tener tienda física y una orden de tienda sí debería tenerla. Los totales coinciden (12.143 órdenes online y 12.143 sin tienda_id), pero eso no garantiza que sean las mismas filas. Además, se revisa que cada tienda_id exista en la tabla de tiendas.

In [21]:
print(pd.crosstab(orders_clean["canal"], orders_clean["tienda_id"].isna(),
                  rownames=["canal"], colnames=["sin tienda_id"]))

tiendas_validas = set(stores["tienda_id"])
con_tienda = orders_clean["tienda_id"].notna()
print("\ntienda_id que no existe en stores:", (con_tienda & ~orders_clean["tienda_id"].isin(tiendas_validas)).sum())

sin tienda_id  False  True 
canal                      
online             0  12143
tienda         10533      0

tienda_id que no existe en stores: 0


**Resultado:** sin hallazgos. Todas las órdenes online (12.143) no tienen tienda_id y todas las de tienda (10.533) sí lo tienen; no hay combinaciones inconsistentes. Todos los tienda_id existen en la tabla de tiendas. Los nulos de tienda_id detectados en la sección 2.1 son esperados: corresponden a las órdenes online.

#### O3. Integridad referencial de clientes

Cada orden con customer_id debería corresponder a un cliente de la tabla customers. Además, los registros de clientes terminan el 2026-06-30 pero las órdenes llegan hasta el 2026-09-30, así que se revisa si hay órdenes de clientes que no están en customers y si se concentran después de junio (lo que indicaría una exportación incompleta de clientes).

In [22]:
ids_clientes = set(customers_clean["customer_id"])
fecha_orden = pd.to_datetime(orders_clean["fecha"], format="%Y-%m-%d")

huerfanas = orders_clean["customer_id"].notna() & ~orders_clean["customer_id"].isin(ids_clientes)
print("Órdenes con customer_id que no existe en customers:", huerfanas.sum())
print("Rango de fechas:", fecha_orden[huerfanas].min().date(), "a", fecha_orden[huerfanas].max().date())
print("Posteriores a junio de 2026:", (fecha_orden[huerfanas] > pd.Timestamp("2026-06-30")).sum())

print("\nÓrdenes sin customer_id por trimestre:")
print(fecha_orden[orders_clean["customer_id"].isna()].dt.to_period("Q").value_counts().sort_index().to_string())

Órdenes con customer_id que no existe en customers: 40
Rango de fechas: 2024-02-25 a 2026-09-02
Posteriores a junio de 2026: 3

Órdenes sin customer_id por trimestre:
fecha
2024Q1     58
2024Q2     45
2024Q3     58
2024Q4     71
2025Q1     72
2025Q2     65
2025Q3    102
2025Q4     76
2026Q1     83
2026Q2     73
2026Q3     97
Freq: Q-DEC


**¿La tabla de clientes está incompleta?** Los registros de clientes terminan el 2026-06-30, pero las órdenes llegan hasta el 2026-09-30. Si faltaran los clientes registrados después de junio, veríamos muchas órdenes huérfanas o sin cliente concentradas después de esa fecha. No es así:
- Solo 3 de las 40 órdenes huérfanas son posteriores a junio de 2026.
- Las 800 órdenes sin customer_id se reparten de forma estable entre trimestres (de 45 a 102). El tercer trimestre es algo mayor tanto en 2025 como en 2026, lo que indica temporada y no un cambio en 2026.

**Conclusión:** las órdenes de julio a septiembre corresponden a clientes ya registrados; la diferencia de fechas no deja clientes por fuera del análisis.

In [23]:
orders_clean["cliente_atribuible"] = orders_clean["customer_id"].isin(ids_clientes)

registrar("O3", "orders", "customer_id que no existe en customers", int(huerfanas.sum()),
          "Se mantienen en el total de ventas y se excluyen del análisis por cliente")

print("Órdenes atribuibles a un cliente:", orders_clean["cliente_atribuible"].sum(), "de", len(orders_clean))

Órdenes atribuibles a un cliente: 21836 de 22676


**Problema:** 40 órdenes tienen un customer_id que no existe en customers, posiblemente por errores de captura puntuales. Sumadas a las 800 órdenes sin cliente, quedan 840 órdenes (3,7%) que no se pueden atribuir a un perfil.

**Tratamiento:** no se eliminan, porque son ventas reales de la marca. Se marcan con cliente_atribuible = False: cuentan en el total de ventas, pero se excluyen del análisis por cliente.

**Impacto si no se corrige:**
- Si se eliminaran, el total de ventas de la marca quedaría subestimado, y el peso de cada segmento ("representa el X% de las ventas") saldría inflado.
- Si se dejaran sin marcar, los 40 IDs huérfanos aparecerían como clientes "fantasma": perfiles sin datos de contacto ni consentimiento, que se colarían en los segmentos y en las listas de campañas.

**Supuesto:** la fecha de corte del análisis es 2026-09-30, la última fecha con órdenes registradas.

#### O4. Fechas de orden

La fecha de la orden define la recencia y la frecuencia de compra, y permite saber si un cliente compró una colección cerca de su lanzamiento. Se revisa el formato, el rango de fechas y si hay órdenes anteriores a la fecha de registro del cliente.

In [24]:
formatos = orders_clean["fecha"].str.replace(r"\d", "9", regex=True)
print("Formatos:", formatos.value_counts().to_dict())

partes = orders_clean["fecha"].str.split("-", expand=True).astype(float)
print(f"Parte del medio máx = {partes[1].max():.0f}, parte final máx = {partes[2].max():.0f}")

Formatos: {'9999-99-99': 22676}
Parte del medio máx = 12, parte final máx = 31


Todas las fechas tienen la forma AAAA-XX-XX, y la parte del medio nunca supera 12: el formato es año-mes-día, igual que en customers. Se convierten de texto a fecha y se revisa el rango y la coherencia con la fecha de registro del cliente.

In [25]:
orders_clean["fecha"] = pd.to_datetime(orders_clean["fecha"], format="%Y-%m-%d", errors="coerce")
print("Fechas no convertidas:", orders_clean["fecha"].isna().sum())
print("Rango:", orders_clean["fecha"].min().date(), "a", orders_clean["fecha"].max().date())

o = orders_clean.merge(customers_clean[["customer_id", "fecha_registro"]], on="customer_id", how="left")
antes = o["fecha"] < o["fecha_registro"]
print("\nÓrdenes anteriores al registro del cliente:", antes.sum())
print("Por canal de la orden:")
print(o.loc[antes, "canal"].value_counts().to_string())
print("Días de diferencia (mediana):", (o.loc[antes, "fecha_registro"] - o.loc[antes, "fecha"]).dt.days.median())

Fechas no convertidas: 0
Rango: 2024-01-01 a 2026-09-30

Órdenes anteriores al registro del cliente: 381
Por canal de la orden:
canal
online    198
tienda    183
Días de diferencia (mediana): 414.0


In [26]:
reg_afectados = o.loc[antes].drop_duplicates("customer_id")["fecha_registro"]
print("Fechas de registro distintas entre los afectados:", reg_afectados.nunique())
print("Por mes de registro:")
print(reg_afectados.dt.to_period("M").value_counts().sort_index().to_string())

Fechas de registro distintas entre los afectados: 136
Por mes de registro:
fecha_registro
2024-01     5
2024-02     1
2024-04     1
2024-05     2
2024-07     3
2024-11     2
2024-12     2
2025-01     6
2025-02     1
2025-05     1
2025-07     4
2025-09     5
2025-10    10
2025-11    12
2025-12    17
2026-01    12
2026-02    14
2026-03    15
2026-04    18
2026-05    16
2026-06    19
Freq: M


No hay fechas inválidas y el rango va de 2024-01-01 a 2026-09-30. Sin embargo, 381 órdenes son anteriores a la fecha de registro de su cliente, repartidas entre online (198) y tienda (183), con una mediana de 414 días de diferencia.

**¿Por qué pasa?** Se descartó una migración de sistema: las fechas de registro de estos clientes son 136 fechas distintas, no una carga masiva en pocos días. Sí hay un patrón: la mayoría se registró desde octubre de 2025, en volumen creciente cada mes. La hipótesis más probable es que son personas que ya compraban (en tienda o sin cuenta online) y que formalizaron su registro después. En ese caso, la fecha de registro no representa el inicio de la relación con la marca.

In [27]:
clientes_afectados = o.loc[antes, "customer_id"].nunique()
print("Clientes afectados:", clientes_afectados)

primera_compra = (orders_clean[orders_clean["cliente_atribuible"]]
                  .groupby("customer_id")["fecha"].min())
customers_clean["primera_compra"] = customers_clean["customer_id"].map(primera_compra)
customers_clean["cliente_desde"] = customers_clean[["fecha_registro", "primera_compra"]].min(axis=1)

registrar("O4", "orders", "Órdenes anteriores a la fecha de registro del cliente", int(antes.sum()),
          "Se conservan las órdenes; la antigüedad del cliente se calcula desde su primera interacción "
          "(la fecha más antigua entre registro y primera compra)")

Clientes afectados: 166


**Problema:** 381 órdenes de X clientes son anteriores a su fecha de registro, con una mediana de más de un año. La fecha de registro de esos clientes no es confiable como inicio de la relación.

**Tratamiento:** las órdenes se conservan, porque son transacciones reales. Se crea cliente_desde, la fecha más antigua entre el registro y la primera compra, para medir la antigüedad real del cliente.

**Impacto si no se corrige:** esos clientes parecerían más nuevos de lo que son. Un cliente con dos años de compras se vería como recién registrado, lo que distorsiona su clasificación en los arquetipos (por ejemplo, "cliente nuevo" vs. "cliente leal").

#### O5. Distribución de estados: qué cuenta como venta

Las órdenes pueden estar completadas, devueltas o canceladas. Antes de calcular ventas hay que definir cuáles cuentan: incluir devoluciones o cancelaciones inflaría las ventas y el valor de los clientes que más devuelven.

In [28]:
print(orders_clean["estado"].value_counts(normalize=True).round(3).to_string())

print("\nEstado por canal (% de las órdenes de cada canal):")
print(pd.crosstab(orders_clean["canal"], orders_clean["estado"], normalize="index").round(3))

orders_clean["es_venta"] = orders_clean["estado"].eq("completada")

estado
completada    0.94
devuelta      0.04
cancelada     0.02

Estado por canal (% de las órdenes de cada canal):
estado  cancelada  completada  devuelta
canal                                  
online      0.018       0.944     0.038
tienda      0.023       0.936     0.041


**Resultado:** el 94% de las órdenes están completadas, el 4% devueltas y el 2% canceladas, con proporciones muy similares entre online y tienda.

#### I1. Líneas duplicadas exactas

Hay 627 filas idénticas en todas sus columnas: mismo order_id, sku, cantidad, precio y descuento. Es normal que una orden tenga varias líneas, pero no que se repitan exactamente iguales. Como no existe un identificador de línea, no se puede distinguir con certeza entre una doble carga y dos líneas legítimas, así que se mide cuánto cambian las ventas en cada caso.

In [29]:
dup = order_items_clean.duplicated()
bruto = order_items_clean["cantidad"] * order_items_clean["precio_unitario"]

print("Filas duplicadas:", dup.sum(), f"({dup.mean():.1%} de las líneas)")
print("Órdenes afectadas:", order_items_clean.loc[dup, "order_id"].nunique())
print("¿Alguna línea aparece más de 2 veces?:", order_items_clean.value_counts().max() > 2)

print(f"\nValor bruto con duplicados: {bruto.sum():,.0f} MXN")
print(f"Valor bruto sin duplicados: {bruto[~dup].sum():,.0f} MXN")
print(f"Diferencia: {bruto[dup].sum():,.0f} MXN ({bruto[dup].sum() / bruto.sum():.1%})")

Filas duplicadas: 627 (1.3% de las líneas)
Órdenes afectadas: 620
¿Alguna línea aparece más de 2 veces?: True

Valor bruto con duplicados: 55,969,915 MXN
Valor bruto sin duplicados: 55,223,099 MXN
Diferencia: 746,816 MXN (1.3%)


Las 627 líneas duplicadas se reparten en 620 órdenes, casi exactamente una por orden afectada. Ese patrón es consistente con una doble carga de líneas y no con compras repetidas del mismo producto, que se concentrarían en pocas órdenes grandes. El valor bruto inflado es de 746.816 MXN (1,3% del total).

In [30]:
antes_filas = len(order_items_clean)
order_items_clean = order_items_clean.drop_duplicates()

registrar("I1", "order_items", "Líneas duplicadas exactas (posible doble carga)", antes_filas - len(order_items_clean),
          "Se eliminan los duplicados exactos; impacto de 746.816 MXN (1,3%) en valor bruto")
print("Líneas:", antes_filas, "->", len(order_items_clean))

Líneas: 48293 -> 47666


**Problema:** 627 líneas de producto están duplicadas exactamente en 620 órdenes, un patrón consistente con una doble carga.

**Tratamiento:** se eliminan los duplicados exactos, conservando una línea de cada uno.

**Impacto si no se corrige:** las ventas se inflarían en 746.816 MXN (1,3%). Aunque es poco a nivel global, el efecto se concentra en 620 órdenes: los clientes de esas órdenes parecerían de mayor valor del que realmente tienen, lo que afecta directamente la pregunta de clientes de mayor valor.

#### I2. Rangos de cantidad, precio y descuento

Que las columnas sean numéricas no garantiza que los valores tengan sentido. Una cantidad negativa o un precio en cero distorsionan las ventas, y un descuento fuera de 0–100, o demasiado alto para el negocio, indica un error de captura o de escala.

In [31]:
print(order_items_clean[["cantidad", "precio_unitario", "descuento_pct"]].describe().round(1).to_string())

print("\nCantidad <= 0:", (order_items_clean["cantidad"] <= 0).sum())
print("Precio <= 0:", (order_items_clean["precio_unitario"] <= 0).sum())
print("Descuento < 0 o > 100:", (~order_items_clean["descuento_pct"].between(0, 100)).sum())

print("\nDistribución de descuentos:")
print(order_items_clean["descuento_pct"].value_counts().sort_index().to_string())

       cantidad  precio_unitario  descuento_pct
count   47666.0          47666.0        47666.0
mean        1.3            983.3            8.5
std         2.3            643.7           14.3
min         1.0            169.0            0.0
25%         1.0            419.0            0.0
50%         1.0            819.0            0.0
75%         1.0           1399.0           15.0
max       120.0           2799.0           50.0

Cantidad <= 0: 0
Precio <= 0: 0
Descuento < 0 o > 100: 0

Distribución de descuentos:
descuento_pct
0     30661
10     4526
15     4687
20     1152
30     1992
40     2367
50     2281


Los precios (169 a 2.799 MXN) y los descuentos son coherentes: los descuentos son escalones típicos de retail (0 a 50%) y no hay valores en cero ni negativos. Sin embargo, la cantidad tiene una mediana de 1 y un máximo de 120 unidades en una sola línea, algo atípico para una marca de moda. Se revisa la distribución de cantidades altas.

In [32]:
cant = order_items_clean["cantidad"]
print(cant.value_counts().sort_index().to_string())

alto = cant > 3
print("\nLíneas con más de 3 unidades:", alto.sum())
print("Órdenes afectadas:", order_items_clean.loc[alto, "order_id"].nunique())

bruto = cant * order_items_clean["precio_unitario"]
print(f"Peso en el valor bruto: {bruto[alto].sum() / bruto.sum():.1%}")

cantidad
1      38922
2       4351
3       4368
50         7
99         7
120       11

Líneas con más de 3 unidades: 25
Órdenes afectadas: 25
Peso en el valor bruto: 4.5%


Las cantidades normales van de 1 a 3 unidades por línea, sin ningún valor intermedio hasta saltar a 50, 99 y 120. Son 25 líneas en 25 órdenes, pero pesan el 4,5% del valor bruto. Para entender su origen se cruzan con los datos de la orden y del cliente.

In [33]:
atipicas = order_items_clean[order_items_clean["cantidad"] > 3]

detalle = (atipicas
           .merge(orders_clean[["order_id", "customer_id", "canal", "tienda_id", "estado"]], on="order_id", how="left")
           .merge(customers_clean[["customer_id", "email"]], on="customer_id", how="left"))

print("Clientes distintos en las 25 líneas:", detalle["customer_id"].nunique())
print("customer_id:", detalle["customer_id"].unique())
print("Email:", detalle["email"].unique())
print("Canal:", detalle["canal"].value_counts().to_dict())

Clientes distintos en las 25 líneas: 1
customer_id: <StringArray>
['C999999']
Length: 1, dtype: str
Email: <StringArray>
['qa.tienda@example.com']
Length: 1, dtype: str
Canal: {'tienda': 25}


Las 25 líneas atípicas pertenecen a un único cliente: C999999, con el email qa.tienda@example.com 

2 señales indican que es una cuenta de prueba y no un cliente real:
- "qa" significa *Quality Assurance*, el equipo que prueba sistemas.
- C999999 es un identificador "comodín", distinto al formato del resto de IDs.

Las cantidades de 50, 99 y 120 unidades son consistentes con pruebas del sistema de punto de venta. Se revisan también las demás cuentas cuyo email no sigue el patrón de los clientes.

In [34]:
email = customers_clean["email"]
es_qa = email.str.startswith("qa.")
es_interna = email.str.endswith("@marcamoda.mx")

print("Emails que no empiezan por 'cliente':", (~email.str.startswith("cliente")).sum())
print("  - cuenta QA:", es_qa.sum())
print("  - dominio de la marca (@marcamoda.mx):", es_interna.sum())

ids_internos = customers_clean.loc[es_interna, "customer_id"]
ord_internas = orders_clean[orders_clean["customer_id"].isin(ids_internos)]
print("\nÓrdenes de cuentas internas:", len(ord_internas))

Emails que no empiezan por 'cliente': 17
  - cuenta QA: 1
  - dominio de la marca (@marcamoda.mx): 16

Órdenes de cuentas internas: 731


De los 17 emails que no siguen el patrón de cliente, 1 es la cuenta de prueba y 16 tienen el dominio de la propia marca (@marcamoda.mx), con 731 órdenes en total. Para entender qué tipo de compras hacen estas cuentas, se analiza su historial orden por orden.

In [35]:
por_orden_int = (order_items_clean
    .merge(ord_internas[["order_id", "customer_id", "fecha", "canal", "tienda_id"]], on="order_id")
    .assign(bruto=lambda d: d["cantidad"] * d["precio_unitario"])
    .groupby(["customer_id", "order_id", "fecha", "canal", "tienda_id"], dropna=False)
    .agg(unidades=("cantidad", "sum"), valor_bruto=("bruto", "sum"), descuento_prom=("descuento_pct", "mean"))
    .reset_index())

resumen_int = (por_orden_int.groupby("customer_id")
    .agg(ordenes=("order_id", "count"), unidades_prom=("unidades", "mean"),
         unidades_max=("unidades", "max"), descuento_prom=("descuento_prom", "mean"))
    .round(1)
    .sort_values("ordenes", ascending=False))

desc_int = order_items_clean.loc[order_items_clean["order_id"].isin(ord_internas["order_id"]), "descuento_pct"].mean()
desc_resto = order_items_clean.loc[~order_items_clean["order_id"].isin(ord_internas["order_id"]), "descuento_pct"].mean()

print("Unidades por orden (promedio por cuenta): de", resumen_int["unidades_prom"].min(), "a", resumen_int["unidades_prom"].max())
print("Máximo de unidades en una sola orden:", resumen_int["unidades_max"].max())
print(f"Descuento promedio — cuentas internas: {desc_int:.1f}% | resto de clientes: {desc_resto:.1f}%")
resumen_int

Unidades por orden (promedio por cuenta): de 1.7 a 2.9
Máximo de unidades en una sola orden: 6
Descuento promedio — cuentas internas: 23.3% | resto de clientes: 8.1%


,ordenes,unidades_prom,unidades_max,descuento_prom
customer_id,,,,
C104001,420,1.8,3,4.8
C101394,29,2.6,6,45.3
C101353,25,2.2,5,44.7
C102322,24,2.5,6,44.7
C102975,24,2.0,3,45.5
C100933,23,1.7,3,45.7
C102745,22,2.7,5,44.8
C102984,22,2.0,3,46.0
C101759,21,2.4,5,45.8


Las cuentas internas compran en promedio entre 1,7 y 2,9 unidades por orden (máximo 6 en una sola orden), como cualquier cliente: no son compras mayoristas. La diferencia está en el descuento: 23,3% en promedio, casi tres veces el 8,1% del resto de clientes. El patrón es consistente con compras de empleados con descuento de personal.

In [36]:
top = resumen_int.index[0]
c = por_orden_int[por_orden_int["customer_id"] == top]

print("Cuenta:", top, "| email:", customers_clean.loc[customers_clean["customer_id"] == top, "email"].iloc[0])
print("Órdenes:", len(c))
print("Canal:", c["canal"].value_counts().to_dict())
print("Tiendas:", c["tienda_id"].value_counts(dropna=False).to_dict())
print("Órdenes por mes (promedio):", round(c.groupby(c["fecha"].dt.to_period("M")).size().mean(), 1))
print("Máximo de órdenes en un mismo día:", c.groupby("fecha").size().max())
print("Días entre compras (mediana):", c["fecha"].sort_values().diff().dt.days.median())

Cuenta: C104001 | email: ventas.mostrador@marcamoda.mx
Órdenes: 420
Canal: {'tienda': 420}
Tiendas: {'T01': 88, 'T02': 85, 'T03': 83, 'T05': 82, 'T04': 82}
Órdenes por mes (promedio): 12.7
Máximo de órdenes en un mismo día: 3
Días entre compras (mediana): 2.0


La cuenta C104001 (ventas.mostrador@marcamoda.mx) concentra 420 de las 731 órdenes internas: todas en tienda física, repartidas casi por igual entre las 5 tiendas (de 82 a 88 cada una), con 12,7 órdenes al mes, una compra cada 2 días y hasta 3 en el mismo día. No corresponde a una persona: es una cuenta genérica de mostrador que se usa en todas las tiendas, como indica su propio email.

In [37]:
# Tamaño de la cuenta de prueba, medido ANTES de eliminarla
id_qa = customers_clean.loc[es_qa, "customer_id"]
ord_qa = orders_clean.loc[orders_clean["customer_id"].isin(id_qa), "order_id"]
lineas_qa = order_items_clean["order_id"].isin(ord_qa)

bruto = order_items_clean["cantidad"] * order_items_clean["precio_unitario"]
print(f"Cuenta QA: {len(ord_qa)} órdenes, {lineas_qa.sum()} líneas, "
      f"{bruto[lineas_qa].sum():,.0f} MXN ({bruto[lineas_qa].sum() / bruto.sum():.1%} del valor bruto)")

# Se elimina la cuenta de prueba con todas sus órdenes y líneas
customers_clean   = customers_clean[~es_qa].copy()
orders_clean      = orders_clean[~orders_clean["order_id"].isin(ord_qa)]
order_items_clean = order_items_clean[~lineas_qa]

# Las cuentas internas se conservan, marcadas
customers_clean["cuenta_interna"] = customers_clean["email"].str.endswith("@marcamoda.mx")

registrar("I2", "order_items", "Cuenta de prueba (QA) con cantidades de 50, 99 y 120 unidades", int(lineas_qa.sum()),
          "Se elimina la cuenta C999999 con todas sus órdenes y líneas; no es un cliente real")
registrar("I2", "customers", "Cuentas internas de la marca (@marcamoda.mx) con descuento de personal", int(es_interna.sum()),
          "Se conservan en el total de ventas y se excluyen de la segmentación de clientes")

print("Clientes:", len(customers_clean), "| Órdenes:", len(orders_clean), "| Líneas:", len(order_items_clean))

Cuenta QA: 25 órdenes, 25 líneas, 2,484,817 MXN (4.5% del valor bruto)
Clientes: 3005 | Órdenes: 22651 | Líneas: 47641


**Problema:** se encontraron dos tipos de cuentas que no son clientes:
- Una **cuenta de prueba** (C999999, qa.tienda@example.com) con 25 órdenes de 50, 99 y 120 unidades, que suman 2.484.817 MXN: el 4,5% del valor bruto de todas las ventas.
- **16 cuentas internas** de la marca (@marcamoda.mx), con 731 órdenes de compras de personal (23,3% de descuento promedio frente al 8,1% del resto). Una de ellas es una cuenta genérica de mostrador con 420 órdenes.

**Tratamiento:** se elimina la cuenta de prueba con todas sus órdenes, porque no son ventas reales. Las cuentas internas se conservan en el total de ventas, porque sí son compras reales, pero se marcan con cuenta_interna y se excluyen de la segmentación: no son clientes a los que el CRM deba dirigirse. Después de la limpieza quedan 3.005 clientes, 22.651 órdenes y 47.641 líneas.

**Impacto si no se corrige:** es el error con mayor efecto económico de todo el análisis. Una sola cuenta de prueba inflaría las ventas en un 4,5% y aparecería como uno de los clientes de mayor valor. La cuenta de mostrador, con 420 órdenes, encabezaría el ranking de clientes por frecuencia sin ser una persona.

**Validaciones sin hallazgos:** no hay precios ni cantidades en cero o negativos; los precios van de 169 a 2.799 MXN y los descuentos son escalones de 0 a 50%.

#### I3. Integridad referencial

Cada línea debe pertenecer a una orden existente y a un producto del catálogo. Una línea sin orden no se puede atribuir a un cliente ni a una fecha, y una línea con un SKU que no está en el catálogo no tiene colección ni fecha de lanzamiento, algo necesario para la pregunta de nuevas colecciones.

In [38]:
sin_orden = ~order_items_clean["order_id"].isin(orders_clean["order_id"])
sin_producto = ~order_items_clean["sku"].isin(products_clean["sku"])

print("Líneas cuyo order_id no existe en orders:", sin_orden.sum())
print("Líneas cuyo sku no existe en products:", sin_producto.sum())
print("SKU distintos sin producto:", order_items_clean.loc[sin_producto, "sku"].nunique())

Líneas cuyo order_id no existe en orders: 0
Líneas cuyo sku no existe en products: 59
SKU distintos sin producto: 58


In [39]:
skus_falt = pd.Series(order_items_clean.loc[sin_producto, "sku"].unique())

num_cat  = products_clean["sku"].str.extract(r"(\d+)")[0].astype(int)
num_falt = skus_falt.str.extract(r"(\d+)")[0].astype(int)

bruto = order_items_clean["cantidad"] * order_items_clean["precio_unitario"]

pd.DataFrame({
    "SKU distintos": [len(num_cat), len(num_falt)],
    "SKU mínimo":    [f"SKU{num_cat.min()}", f"SKU{num_falt.min()}"],
    "SKU máximo":    [f"SKU{num_cat.max()}", f"SKU{num_falt.max()}"],
    "Líneas vendidas": [(~sin_producto).sum(), sin_producto.sum()],
    "% valor bruto": [f"{bruto[~sin_producto].sum() / bruto.sum():.2%}", f"{bruto[sin_producto].sum() / bruto.sum():.2%}"],
}, index=["En catálogo", "Fuera del catálogo"])

,SKU distintos,SKU mínimo,SKU máximo,Líneas vendidas,% valor bruto
En catálogo,240,SKU1001,SKU1240,47582,99.88%
Fuera del catálogo,58,SKU9056,SKU9988,59,0.12%


Los SKU que faltan no son errores de escritura: pertenecen a un rango distinto al del catálogo (SKU9xxx frente a SKU1001–SKU1240) y no coinciden con ningún producto. Al cruzarlos con las órdenes y los clientes no aparece ningún patrón (no se concentran en un cliente, canal ni periodo). Lo más probable es que sean productos descontinuados o que no se incluyeron en la exportación del catálogo.

In [40]:
order_items_clean["sku_en_catalogo"] = order_items_clean["sku"].isin(products_clean["sku"])

registrar("I3", "order_items", "SKU que no existe en el catálogo (rango SKU9xxx)", int(sin_producto.sum()),
          "Se conservan en las ventas; se excluyen solo del análisis de colecciones por no tener colección asignada")

print("Líneas fuera del catálogo marcadas:", (~order_items_clean["sku_en_catalogo"]).sum())

Líneas fuera del catálogo marcadas: 59


**Problema:** 59 líneas de venta tienen 58 SKU que no existen en el catálogo, por lo que no tienen colección, categoría ni fecha de lanzamiento.

**Tratamiento:** se conservan en las ventas, porque son compras reales, y se marcan con sku_en_catalogo = False. Solo se excluyen del análisis de nuevas colecciones.

**Impacto si no se corrige:** al unir las ventas con el catálogo, estas líneas quedarían sin colección. Según el tipo de unión, se perderían de las ventas o aparecerían como una colección "vacía" en los resultados. Su peso es bajo (0,12% del valor bruto), por lo que no cambia las conclusiones.

#### I4. Precio unitario vs. precio de lista

Antes de calcular ventas netas hay que saber si precio_unitario es el precio de lista (y el descuento se aplica aparte) o si ya tiene el descuento incluido. Si ya lo incluye y se vuelve a restar descuento_pct, las ventas quedarían subestimadas. También se revisa que no haya precios muy alejados del precio de lista.

In [41]:
cmp = order_items_clean.merge(products_clean[["sku", "precio_lista"]], on="sku", how="inner")

cmp["ratio"] = cmp["precio_unitario"] / cmp["precio_lista"]
cmp["ratio_con_desc"] = 1 - cmp["descuento_pct"] / 100

print("Líneas donde precio_unitario = precio_lista:",
      f"{(cmp['ratio'].round(3) == 1).mean():.1%}")
print("Líneas donde precio_unitario = precio_lista con descuento ya aplicado:",
      f"{(cmp['ratio'].round(3) == cmp['ratio_con_desc'].round(3)).mean():.1%}")

print("\nRatio precio_unitario / precio_lista:")
print(cmp["ratio"].describe().round(3).to_string())

Líneas donde precio_unitario = precio_lista: 100.0%
Líneas donde precio_unitario = precio_lista con descuento ya aplicado: 64.3%

Ratio precio_unitario / precio_lista:
count    47582.0
mean         1.0
std          0.0
min          1.0
25%          1.0
50%          1.0
75%          1.0
max          1.0


**Resultado:** en el 100% de las líneas, precio_unitario es igual al precio de lista del catálogo, así que el descuento **no** está incluido en el precio. (El 64,3% de coincidencia con el precio con descuento corresponde a las líneas sin descuento, donde ambos valores son iguales.) Tampoco hay precios alejados del precio de lista.

**Definición:** la venta neta de cada línea se calcula como cantidad × precio_unitario × (1 − descuento_pct / 100).

**Impacto si no se verifica:** si se asumiera que el precio ya incluía el descuento, las ventas se sobrestimarían, porque se ignorarían todos los descuentos. Si se asumiera lo contrario sin que fuera cierto, se descontaría dos veces y las ventas quedarían subestimadas.

#### P1. Fecha de lanzamiento

fecha_lanzamiento es la base de la pregunta de nuevas colecciones: define cuándo un producto es "nuevo". Se revisa su formato (igual que en las otras tablas) y los 24 productos sin fecha detectados en la sección 2.1.

In [42]:
fl = products_clean["fecha_lanzamiento"]

print("Formatos:", fl.str.replace(r"\d", "9", regex=True).value_counts(dropna=False).to_dict())
partes = fl.str.split("-", expand=True).astype(float)
print(f"Parte del medio máx = {partes[1].max():.0f}, parte final máx = {partes[2].max():.0f}")

products_clean["fecha_lanzamiento"] = pd.to_datetime(fl, format="%Y-%m-%d", errors="coerce")
print("\nFechas no convertidas:", products_clean["fecha_lanzamiento"].isna().sum() - fl.isna().sum())
print("Rango:", products_clean["fecha_lanzamiento"].min().date(), "a", products_clean["fecha_lanzamiento"].max().date())

print("\nColección de los productos sin fecha:")
print(products_clean.loc[products_clean["fecha_lanzamiento"].isna(), "coleccion"].value_counts().to_string())

Formatos: {'9999-99-99': 216, nan: 24}
Parte del medio máx = 8, parte final máx = 20

Fechas no convertidas: 0
Rango: 2024-02-15 a 2026-08-18

Colección de los productos sin fecha:
coleccion
BASICOS    24


**Resultado:** las 216 fechas tienen la forma AAAA-XX-XX. Como la parte final llega hasta 20 (y un mes nunca supera 12), la parte final es el día: el formato es año-mes-día, igual que en las otras tablas. Las fechas se convirtieron sin errores y van de 2024-02-15 a 2026-08-18. Los 24 productos sin fecha son todos BASICOS, que por definición no pertenecen a una temporada, así que no es un error.

#### P2. Consistencia de colecciones

Cada colección (PV = primavera-verano, OI = otoño-invierno, más el año) debería tener sus productos lanzados en una misma época del año correspondiente. Se revisa que los nombres de colección y categoría sean consistentes y que las fechas de lanzamiento cuadren con la colección.

In [43]:
print("Categorías:", products_clean["categoria"].value_counts().to_dict())

resumen_col = (products_clean.groupby("coleccion")
               .agg(productos=("sku", "count"),
                    lanzamiento_min=("fecha_lanzamiento", "min"),
                    lanzamiento_max=("fecha_lanzamiento", "max"))
               .sort_values("lanzamiento_min"))
resumen_col

Categorías: {'Vestidos': 36, 'Blusas': 36, 'Pantalones': 36, 'Chamarras': 36, 'Calzado': 36, 'Accesorios': 36, 'Playeras básicas': 8, 'Jeans básicos': 8, 'Ropa interior': 8}


,productos,lanzamiento_min,lanzamiento_max
coleccion,,,
PV24,36,2024-02-15,2024-02-15
OI24,36,2024-08-20,2024-08-20
PV25,36,2025-02-13,2025-02-13
OI25,36,2025-08-19,2025-08-19
PV26,36,2026-02-12,2026-02-12
OI 26,1,2026-08-18,2026-08-18
OI-26,3,2026-08-18,2026-08-18
OI26,28,2026-08-18,2026-08-18
OI26,1,2026-08-18,2026-08-18


Cada colección tiene 36 productos lanzados en un mismo día (mediados de febrero para PV, mediados de agosto para OI), y las categorías están escritas de forma consistente. Sin embargo, la colección más reciente, OI26, aparece escrita de 6 formas distintas ("OI26", "OI-26", "oi26", "OI 26", "Oi26" y una con espacios), aunque todas comparten la misma fecha de lanzamiento (2026-08-18) y suman 36 productos como las demás.

In [44]:
antes = products_clean["coleccion"].nunique()
mal_escritas = (products_clean["coleccion"] != "OI26") & products_clean["coleccion"].str.upper().str.replace(r"[\s-]", "", regex=True).eq("OI26")

products_clean["coleccion"] = products_clean["coleccion"].str.upper().str.replace(r"[\s-]", "", regex=True)

registrar("P2", "products", "Colección OI26 escrita de 6 formas distintas", int(mal_escritas.sum()),
          "Se normaliza a mayúsculas, sin espacios ni guiones")

print("Colecciones distintas:", antes, "->", products_clean["coleccion"].nunique())
print(products_clean["coleccion"].value_counts().to_string())

Colecciones distintas: 12 -> 7
coleccion
PV24       36
OI24       36
PV25       36
OI25       36
PV26       36
OI26       36
BASICOS    24


**Problema:** 8 de los 36 productos de OI26 (22%) tenían la colección escrita con otra capitalización, espacios o guiones.

**Tratamiento:** se normalizó el nombre de todas las colecciones (mayúsculas, sin espacios ni guiones). OI26 queda con sus 36 productos.

**Impacto si no se corrige:** OI26 es la colección más reciente, la base para responder quiénes se interesan por las nuevas colecciones. Sin la corrección, el 22% de sus productos y todas sus ventas quedarían fuera de ese análisis, y los clientes que compraron esos productos no se identificarían como interesados en lo nuevo.

#### X1. Órdenes sin ítems

Una orden sin ninguna línea en order_items no tiene productos ni valor. Si se cuenta igual, infla la frecuencia de compra del cliente (parecería que compró más veces) sin aportar ventas.

In [45]:
sin_items = ~orders_clean["order_id"].isin(order_items_clean["order_id"])

print("Órdenes sin ítems:", sin_items.sum(), f"({sin_items.mean():.1%})")
print("Por estado:", orders_clean.loc[sin_items, "estado"].value_counts().to_dict())
print("Por canal:", orders_clean.loc[sin_items, "canal"].value_counts().to_dict())

Órdenes sin ítems: 0 (0.0%)
Por estado: {}
Por canal: {}


**Resultado:** sin hallazgos. Todas las órdenes tienen al menos una línea en order_items, así que no hay órdenes vacías que inflen la frecuencia de compra.

### 2.4 Resumen de calidad de datos

Tabla con todos los problemas encontrados, cuántos registros afectaban y cómo se trataron.

In [46]:
resumen_calidad = pd.DataFrame(registro)
resumen_calidad

,codigo,tabla,problema,filas_afectadas,accion
0,C1,customers,Misma persona registrada en tienda y online co...,116,Se conserva el registro más antiguo y se toman...
1,C2,customers,"Email con formato inválido (sin @, doble @ o s...",8,No se corrige para no adivinar el contacto; se...
2,C4,customers,Fecha de nacimiento imposible (valor por defec...,42,Se reemplaza por nulo; no se usa la edad en es...
3,C5,customers,Teléfono sin prefijo de país (10 dígitos),48,Se estandariza agregando el prefijo 52
4,C5,customers,Teléfono incompleto (2 o 4 dígitos),8,Se reemplaza por nulo y se marca como no conta...
5,C6,customers,Texto con forma de instrucción de negocio dent...,3,No se aplica; se documenta como hallazgo y la ...
6,O3,orders,customer_id que no existe en customers,40,Se mantienen en el total de ventas y se excluy...
7,O4,orders,Órdenes anteriores a la fecha de registro del ...,381,Se conservan las órdenes; la antigüedad del cl...
8,I1,order_items,Líneas duplicadas exactas (posible doble carga),627,Se eliminan los duplicados exactos; impacto de...
9,I2,order_items,"Cuenta de prueba (QA) con cantidades de 50, 99...",25,Se elimina la cuenta C999999 con todas sus órd...


### 2.5 Exportación de datos limpios

Las tablas limpias se guardan en data/processed/. Esta carpeta está dentro de data/, excluida del repositorio, porque los datos limpios siguen conteniendo información personal.

In [47]:
SALIDA = DATA / "processed"
SALIDA.mkdir(exist_ok=True)

customers_clean.to_csv(SALIDA / "customers.csv", index=False)
orders_clean.to_csv(SALIDA / "orders.csv", index=False)
order_items_clean.to_csv(SALIDA / "order_items.csv", index=False)
products_clean.to_csv(SALIDA / "products.csv", index=False)
resumen_calidad.to_csv(SALIDA / "resumen_calidad.csv", index=False)

print("Exportado:", [f.name for f in SALIDA.glob("*.csv")])

Exportado: ['customers.csv', 'orders.csv', 'order_items.csv', 'products.csv', 'resumen_calidad.csv']
